# Graph retrievers — interactive tour

Three `BaseRetriever`s over one AgensGraph store, the examples that teach the query
generator this engine's dialect, the index awareness that keeps a generated query off a
full label read, and the eval that holds all of it in place.

Every film, person and plot in this catalog is **invented**, so the model answering has
nothing to lean on but what each retriever hands it — every difference below is
retrieval, not recall.

**Prerequisites**

```bash
cd langchain
.venv/bin/python examples/demos/05_graph_retrievers/ingest.py    # this notebook's catalog
.venv/bin/python examples/demos/01_arxiv_graphrag/prepare.py     # optional, for the 138k-vertex leg
```

and `AGENSGRAPH_UNTRUSTED_URL` set to the `llm_ro` role (see the demos README, "Roles"):
model-written Cypher runs as that role. A handful of OpenAI calls run below (embeddings +
`gpt-4o-mini`), together well under a cent; `DEMOS_FAKE=1` runs them on the fake models.

In [ ]:
import json
import pathlib
import sys

HERE = pathlib.Path.cwd()                       # .../05_graph_retrievers
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                                          # demos root, for _common
sys.path.insert(0, str(p.parent.parent / "evals" / "text2cypher"))   # the eval's own module

import pandas as pd
from _common import agens, config, models

from langchain_agensgraph import log_queries
from langchain_agensgraph.chains import DIALECT_EXAMPLES, AgensCypherQAChain
from langchain_agensgraph.retrievers import (
    AgensGraphContextRetriever,
    AgensText2CypherRetriever,
    AgensVectorRetriever,
    render_graph_context,
)

GRAPH = "movie_retrievers"
LIGHTHOUSE = "divers find a lighthouse that is somehow still burning"

url = config.url()
config.require_untrusted_url()
llm = models.get_llm()
store = agens.make_vector(models.get_embeddings(), graph_name=GRAPH,
                          label="Movie", text_property="plot")
graph = agens.make_graph(GRAPH)
agens.grant(GRAPH)  # render_grants, run as the owner: llm_ro may read the graph


def docs_df(docs, chars=46):
    return pd.DataFrame([{
        "title": d.metadata.get("title"),
        "year": d.metadata.get("year"),
        "score": round(d.metadata["score"], 3) if "score" in d.metadata else None,
        "neighbours": len(d.metadata.get("_context_nodes_", [])) or None,
        "edges": len(d.metadata.get("_context_rels_", [])) or None,
        "page_content": d.page_content[:chars].replace("\n", " ") + "…",
    } for d in docs])


print(f"{url.split('@')[-1]} · graph {GRAPH}")

## The catalog

Eight films, their directors and cast, and their genres — movies carry the embedded plot
text, everyone else is ordinary graph structure around them.

In [2]:
vertices = graph.query("MATCH (n) RETURN label(n) AS label, count(*) AS n ORDER BY n DESC")
edges = graph.query("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY n DESC")
print("vertices:", {r["label"]: r["n"] for r in vertices})
print("edges:   ", {r["type"]: r["n"] for r in edges})

pd.DataFrame(graph.query(
    'MATCH (p:person)-[:directed]->(m:"Movie") '
    "RETURN m.title AS title, m.year AS year, p.name AS director ORDER BY year"))

vertices: {'person': 10, 'Movie': 8, 'genre': 6}
edges:    {'acted_in': 16, 'in_genre': 14, 'directed': 8}


,title,year,director
0,Winter Arithmetic,2017,Ines Varga
1,Marrow and Vine,2018,Sana Whitfield
2,The Salt Meridian,2019,Ines Varga
3,The Lantern Divers,2020,Sana Whitfield
4,Glasshouse Protocol,2021,Teodor Malik
5,The Eighth Ferry,2022,Ruth Okonjo
6,Quiet Cartography,2023,Teodor Malik
7,Static Bloom,2024,Ruth Okonjo


## 1 · `AgensVectorRetriever` — one similarity search, scored

The store's vector search (or its hybrid fusion) as a LangChain retriever. Each document
carries its score in `metadata["score"]`: for a vector search the cosine distance, smaller
being nearer.

In [ ]:
vector = AgensVectorRetriever(vectorstore=store, k=3)
docs_df(vector.invoke(LIGHTHOUSE))

`k` and the search arguments a call passes to `invoke` — `filter`, `fetch_k`, a retrieval
query — win over the retriever's own.

In [4]:
print("constructor k=3 →", len(vector.invoke(LIGHTHOUSE)), "documents")
print("invoke k=1      →", len(vector.invoke(LIGHTHOUSE, k=1)), "document")

# A metadata filter reaches the database as a predicate, not a post-filter.
docs_df(vector.invoke("a quiet mystery", k=3, filter={"year": {"$gte": 2022}}))

constructor k=3 → 3 documents


invoke k=1      → 1 document


,title,year,score,neighbours,edges,page_content
0,Quiet Cartography,2023,0.353,None,None,A retired spy maps the silences in old wiretap…
1,Static Bloom,2024,0.254,None,None,A radio astronomer falls for the voice inside …
2,The Eighth Ferry,2022,0.236,None,None,A night-shift ferry pilot realizes her seven s…


## 2 · `AgensGraphContextRetriever` — the seed *and* its neighbourhood

A plot never names its director. This retriever runs the same similarity search, then
walks out from every hit at once and brings each one's graph neighbourhood back,
structured in metadata: `_context_nodes_` and `_context_rels_`.

In [ ]:
context = AgensGraphContextRetriever(vectorstore=store, k=1, hops=1)
doc = context.invoke(LIGHTHOUSE)[0]
print(doc.page_content, "\n")

pd.DataFrame([{"label": n["label"],
               "name": n["properties"].get("name") or n["properties"].get("title")}
              for n in doc.metadata["_context_nodes_"]])

`render_graph_context` is a ready-made `document_formatter` that writes that
neighbourhood under the seed's own text, for handing straight to a prompt.

In [ ]:
rendered = AgensGraphContextRetriever(vectorstore=store, k=1, hops=2,
                                      document_formatter=render_graph_context)
print(rendered.invoke(LIGHTHOUSE)[0].page_content)

Hops widen the reach (up to 3); a single relationship type narrows it; the caps bound what
each document carries, and how many edges are read from any one node, so a hub cannot
flood the result.

In [ ]:
rows = []
for hops in (1, 2, 3):
    d = AgensGraphContextRetriever(vectorstore=store, k=1, hops=hops).invoke(LIGHTHOUSE)[0]
    rows.append({"setting": f"hops={hops}",
                 "neighbours": len(d.metadata["_context_nodes_"]),
                 "edges": len(d.metadata["_context_rels_"])})

d = AgensGraphContextRetriever(vectorstore=store, k=1, hops=2,
                               relationship_type="directed").invoke(LIGHTHOUSE)[0]
rows.append({"setting": 'hops=2 · relationship_type="directed"',
             "neighbours": len(d.metadata["_context_nodes_"]),
             "edges": len(d.metadata["_context_rels_"])})

d = AgensGraphContextRetriever(vectorstore=store, k=1, hops=2, max_context_nodes=3,
                               max_context_rels=3).invoke(LIGHTHOUSE)[0]
rows.append({"setting": "hops=2 · caps=3", "neighbours": len(d.metadata["_context_nodes_"]),
             "edges": len(d.metadata["_context_rels_"])})

pd.DataFrame(rows)

### Two statements, whatever the fan-out

The seed search is one statement; the walk is one more, from every seed at once, so four
seeds and every neighbour within two hops arrive in two round trips — never one query per
seed. The driver reports every statement it sends, so the claim is checkable rather than
promised: below, the search (after the one setting it needs) and the walk.

In [ ]:
wide = AgensGraphContextRetriever(vectorstore=store, k=4, hops=2)
wide.invoke(LIGHTHOUSE)                      # warm the pooled connections first

with log_queries() as sent:
    docs = wide.invoke(LIGHTHOUSE)

neighbours = sum(len(d.metadata["_context_nodes_"]) for d in docs)
print(f"{len(docs)} documents · {neighbours} neighbours · {len(sent)} record(s) logged\n")
display(pd.DataFrame([{
    "ms": round(one.elapsed * 1000, 2),
    "rows": one.rows,
    "statement": one.statement[:60].replace("\n", " ").strip(),
} for one in sent]))

print(next(one.statement for one in sent if one.statement.startswith("WITH RECURSIVE")))

## 3 · What the graph context buys

The same question, the same model, two retrievers.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ANSWER = ChatPromptTemplate.from_messages([
    ("system", "Answer from the context alone. If it does not contain the answer, say so plainly."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])


def answer(retriever, question):
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    return (ANSWER | llm).invoke({"context": context, "question": question}).content


question = "Who directed the film about divers finding a lighthouse that still burns?"
print("vector only     :", answer(AgensVectorRetriever(vectorstore=store, k=2), question))
print()
print("+ graph context :", answer(AgensGraphContextRetriever(
    vectorstore=store, k=2, hops=2, document_formatter=render_graph_context), question))

## 4 · `AgensText2CypherRetriever` — the model writes the query, a role made for it runs it

Counting is a query, not a similarity. The generated Cypher runs as the untrusted role
(`llm_ro`), and travels in each document's metadata, so what ran is always visible.

In [ ]:
scribe = AgensText2CypherRetriever(
    graph=graph, llm=llm, k=5,
    # Model-written Cypher runs as the engine's untrusted role; this says so.
    allow_dangerous_requests=True,
)
docs = scribe.invoke("How many films did each director make? Give the name and the count.")
print(docs[0].metadata["cypher"], "\n")
pd.DataFrame([json.loads(d.page_content) for d in docs])

### The boundary is a role, not a regular expression

`query_untrusted` reads the text first and refuses anything that is not plainly one Cypher
read, before a connection is taken. Reading is a fast first opinion, and not the whole
boundary: this is PostgreSQL underneath, where `INSERT`, `TRUNCATE` and `COPY … TO PROGRAM`
are all available and none of them is Cypher. What contains a statement is the role it runs
as — `llm_ro`, which `render_grants` gives `SELECT` on the graph's labels and nothing else —
in a read-only transaction that is always rolled back. From 2.16 the server also refuses a
Cypher write there itself; on 2.14 and 2.15 it does not, so the role is checked to hold no
write privilege at all.

In [ ]:
from langchain_agensgraph.errors import UnsafeCypherError
from langchain_agensgraph.roles import render_grants

print(render_grants(GRAPH))                  # what agens.grant ran, as the owner

for candidate in ('MATCH (m:"Movie") RETURN m.title AS title LIMIT 2',
                  'MATCH (m:"Movie") DETACH DELETE m',
                  'MATCH (m:"Movie") SET m.year = 1900'):
    try:
        print(f"ran      · {candidate} → {graph.query_untrusted(candidate)}")
    except UnsafeCypherError as exc:
        print(f"refused  · {candidate} → {str(exc).splitlines()[0][:80]}")
print("\nmovies still   :", graph.query('MATCH (m:"Movie") RETURN count(*) AS n')[0]["n"])

### Self-correction, for the query that runs clean and matches nothing

A relationship written against its schema direction plans, runs, and returns zero rows.
There is no error to retry on, so `retry_on_empty` treats the silence itself as feedback
and asks again — measured live below against the 138k-vertex arXiv graph, where an
abstract never names its authors and `AUTHORED_BY` is the direction a model tends to
flip.

In [ ]:
try:
    arxiv = agens.make_graph("arxiv")
    papers = arxiv.query('MATCH (p:"Paper") RETURN count(*) AS n')[0]["n"]
except Exception as exc:
    arxiv, papers = None, 0
    print("arxiv graph unavailable:", str(exc).splitlines()[0][:70])

if papers:
    print(f"{papers:,} papers\n")
    agens.grant("arxiv")
    scholar = AgensText2CypherRetriever(graph=arxiv, llm=llm, k=3, max_retries=2,
                                        retry_on_empty=True, allow_dangerous_requests=True)
    docs = scholar.invoke("Which author has written the most papers?")
    print(docs[0].metadata["cypher"], "\n")
    display(pd.DataFrame([json.loads(d.page_content) for d in docs]))
else:
    print("run examples/demos/01_arxiv_graphrag/prepare.py for this leg")

## 5 · Teaching the dialect by example

`DIALECT_EXAMPLES` is a pack of question-and-query pairs, each one there because a model
reaching for its habits gets that case wrong here — labels fold to lower case unless
double-quoted, `count()` takes an argument or `*`, a graph id is `labid.locid`,
`substring` starts at zero. Passed as `examples=`, they become real conversation turns
ahead of the question: each one shown answered with the query alone, which is the shape
the model is being asked to produce.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class SentPrompts(BaseCallbackHandler):
    """Keeps the messages each chat model call is given."""

    def __init__(self):
        self.sent = []

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.sent.append(messages[0])


pd.set_option("display.max_colwidth", 78)
display(pd.DataFrame(DIALECT_EXAMPLES, columns=["question", "query"]).head(8))

taught = AgensCypherQAChain.from_llm(llm, graph=graph, top_k=10,
                                     examples=DIALECT_EXAMPLES, allow_dangerous_requests=True)
seen = SentPrompts()
taught.run_cypher("How many films are there?", config={"callbacks": [seen]})
turns = seen.sent[0]                          # what the Cypher-writing call was given
print(f"{len(turns)} messages · " + " → ".join(m.type for m in turns[:6]) + " → …")

## 6 · Index awareness

A predicate an index cannot serve returns the right rows and reads the whole label;
nothing but the plan says so. The first half of the answer is telling the model which
indexes exist — the schema it already reads now names them.

In [14]:
# Start from a graph with no property indexes, so this reads the same on a re-run.
for name in ("movie_title", "movie_year", "person_name"):
    graph.query(f"DROP PROPERTY INDEX IF EXISTS {name}")
graph.refresh_schema(force=True)
print("index section before:", "Property indexes" in graph.get_schema)

for ddl in ('CREATE PROPERTY INDEX IF NOT EXISTS movie_title ON "Movie" (title)',
            'CREATE PROPERTY INDEX IF NOT EXISTS movie_year ON "Movie" (year)',
            "CREATE PROPERTY INDEX IF NOT EXISTS person_name ON person (name)"):
    graph.query(ddl)
graph.refresh_schema(force=True)

print("index section after :", "Property indexes" in graph.get_schema, "\n")
lines = [ln.strip() for ln in graph.get_schema.strip().splitlines() if ln.strip()]
print("\n".join(lines[-2:]))

index section before: False
index section after : True 

Property indexes are the following (and id(x) is always indexed, on every label):
['(:"Movie") ON (title)', '(:"Movie") ON (year)', '(:"person") ON (name)']


### What those indexes can and cannot serve — measured here, now

`plan_has_index_path` (from the eval's own module) EXPLAINs a statement with
`enable_seqscan = off`: a plan that still carries `Disabled: true` is the planner's own
statement that **no index path exists at all** — as opposed to one it merely declined on
cost. Each pair below returns identical rows either way.

In [ ]:
import t2c                       # the eval's module: dataset, result canon, plan probe

pairs = [
    ("prefix",
     "MATCH (m:\"Movie\") WHERE m.title STARTS WITH 'The' RETURN m.title AS title",
     "MATCH (m:\"Movie\") WHERE m.title >= 'The' AND m.title < 'Thf' RETURN m.title AS title"),
    ("arithmetic",
     'MATCH (m:"Movie") WHERE m.year + 0 > 2020 RETURN m.title AS title',
     'MATCH (m:"Movie") WHERE m.year > 2020 RETURN m.title AS title'),
    ("negation",
     'MATCH (m:"Movie") WHERE m.year <> 2019 RETURN m.title AS title',
     "MATCH (m:\"Movie\") WHERE m.year IN [2017, 2018, 2020, 2021, 2022, 2023, 2024] "
     "RETURN m.title AS title"),
]

rows = []
for case, habit, rewrite in pairs:
    for spelling, cypher in (("habit", habit), ("rewrite", rewrite)):
        rows.append({
            "case": case,
            "spelling": spelling,
            "rows": len(graph.query(cypher)),
            "index path": t2c.plan_has_index_path(url, GRAPH, cypher),
            "predicate": cypher.split("WHERE ")[1].split(" RETURN")[0][:54],
        })
pd.DataFrame(rows)

### Does the generator take the hint?

Three dialect-sensitive questions, generated with and without the example pack, judged by
the same plan probe. The rules live in the system prompt either way, so this is mostly a
check on *them* — and on where they still fall short: asked for "not 2019", both prompts
reach for `<>`, which answers correctly off a full label read. That is the one recurring
miss the eval reports too.

In [ ]:
from langchain_agensgraph.errors import AgensQueryError

plain = AgensCypherQAChain.from_llm(llm, graph=graph, top_k=10, allow_dangerous_requests=True)
questions = ["Which movie titles start with The?",
             "Which movies are not from 2019? Give their titles.",
             "Which film is titled exactly Static Bloom?"]

rows = []
for question in questions:
    for name, chain in (("0-shot", plain), ("few-shot", taught)):
        try:
            cypher = chain.run_cypher(question).cypher   # written, and run as llm_ro
            served = t2c.plan_has_index_path(url, GRAPH, cypher)
        except (UnsafeCypherError, AgensQueryError) as exc:
            cypher, served = str(exc), None            # the corrections ran out
        rows.append({
            "question": question[:36],
            "prompt": name,
            "index path": served,
            "generated predicate": (cypher.split("WHERE ")[1][:52] if "WHERE " in cypher
                                    else cypher[:52]),
        })
pd.DataFrame(rows)

## 7 · The eval that holds all of this in place

The dataset is 99 question-and-gold-query pairs in this dialect, most of them carrying
the query a model writes elsewhere together with the outcome that makes the trap real:
it errors, it returns different rows, or — the silent case — it returns *exactly* the
right rows off a full label read. None of that is asserted in prose: the gold queries,
the habit queries and the plan claims all execute.

In [ ]:
dataset = t2c.load_dataset()
print(f"{len(dataset)} entries")
display(pd.Series([e["category"] for e in dataset]).value_counts().to_frame("entries").T)

# The dataset's own fixture graphs, built by the module that ships beside it.
for name in t2c.FIXTURE_GRAPHS:
    fixture = agens.make_graph(name, create=True)
    t2c.BUILDERS[name](fixture)
    fixture.close()

rows = []
for entry in [e for e in dataset if e.get("habit_outcome") == "unindexed"]:
    fixture = agens.make_graph(entry["graph"])
    gold, habit = fixture.query(entry["gold"]), fixture.query(entry["habit"])
    rows.append({
        "entry": entry["id"],
        "same rows": t2c.results_match(habit, gold, entry.get("ordered", False)),
        "gold index path": t2c.plan_has_index_path(url, entry["graph"], entry["gold"]),
        "habit index path": t2c.plan_has_index_path(url, entry["graph"], entry["habit"]),
    })
    fixture.close()
pd.DataFrame(rows)

### Scoring a model against it

```bash
cd langchain
.venv/bin/python evals/text2cypher/run.py --graphs t2c_movies,t2c_traps,arxiv
.venv/bin/python evals/text2cypher/run.py --examples 17 --retry --graphs t2c_movies,t2c_traps,arxiv
```

Measured on `gpt-4o-mini` over all 99 entries against 2.18-devel:

| configuration    | executable | execution accuracy | index-served |
| ---------------- | ---------- | ------------------ | ------------ |
| 0-shot           | 90%        | 60%                | 92%          |
| 0-shot + retry   | 94%        | 68%                | 92%          |
| 17-shot          | 83%        | 66%                | 92%          |
| 17-shot + retry  | 90%        | **73%**            | **92%**      |

The index rules and the schema's index section carry the index-served rate on their own —
it is 12/13 in every configuration, and the recurring miss is the honest kind: asked for
"not 2019", the model writes `<>`, which answers correctly off a full label read. Details
and the per-item files: `evals/text2cypher/README.md`.

In [18]:
store.close()
graph.close()
if arxiv is not None:
    arxiv.close()
agens.close()
print("connections closed")

connections closed
